In [ ]:
import sys
import os
from dotenv import load_dotenv

# Adds the parent directory (project root) to the python path
sys.path.append(os.path.abspath(os.path.join("../..")))
load_dotenv()

In [ ]:
import pandas as pd

questions_df = pd.read_parquet('../../data/raw/confluence_questions.parquet')

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
import torch

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_NAME    = "all-MiniLM-L6-v2"  
DEVICE        = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

# ── Load model once ───────────────────────────────────────────────────────────
model = SentenceTransformer(MODEL_NAME, device=DEVICE) # type: ignore
query_vector: list[float] = model.encode("Do I need approval to add a large planter or wall mounted art in a Redwood office, and what details should I include in the request?").tolist()
client = chromadb.PersistentClient(path="../../data/processed/embeddings")
collection = client.get_collection(name="c_sem")

results = collection.query(
    query_embeddings=[query_vector], 
    n_results=1,
    where={"doc_id": "dsid_bb14e2f3925b4e27829c4f3c1aea3af1"}                                             
)

# results = collection.get(
#     ids=["c_sem_b9e57264-0889-498a-94bf-53e6b8aea944"]
# )

# 4. Extract and print the sample answer document
if results["documents"] and len(results["documents"][0]) > 0:
    sample_answer = results["documents"][0][0]
    id = results["ids"][0][0] # type: ignore
    metadata = results["metadatas"][0][0]['doc_id'] # type: ignore
    print(f"Sample Answer: {sample_answer}")
    print(f"Chunk ID: {id}")
    # print(f"Metadata: {metadata}")
else:
    print("No matching document found.")

In [ ]:
import json
from chromadb.api import ClientAPI
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

# ──────────────────────────────────────────────
# CONFIG  ← edit these two lines
# ──────────────────────────────────────────────
CHROMA_PATH      = "../../data/processed/embeddings" 
PARQUET_PATH     = "../../data/raw/confluence_questions.parquet" 
OUTPUT_PATH      = "../../data/processed/questions/questions_with_chunk_ids.parquet"

COLLECTION_NAMES = ["c_512", "c_1024", "c_rec", "c_sem"]
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"

# Similarity threshold for semantic fallback (cosine distance).
SIMILARITY_THRESHOLD = 0.6
# ──────────────────────────────────────────────


def load_collections(client: ClientAPI) -> dict:
    """Load all four ChromaDB collections into a dict keyed by name."""
    collections = {}
    for name in COLLECTION_NAMES:
        try:
            collections[name] = client.get_collection(name)
            print(f"Loaded collection: {name}  "
                  f"({collections[name].count()} chunks)")
        except Exception as e:
            print(f"Could not load collection '{name}': {e}")
    return collections


def resolve_chunk_ids_for_question(
    answer: str,
    doc_ids: list[str],
    collections: dict,
    embed_model: SentenceTransformer,
) -> dict:
    """
    For a single question, find the best matching chunk per (doc_id, collection).
 
    For each doc_id:
      - Embed the ground_truth_answer
      - Query each collection filtered by where={"doc_id": doc_id}
      - Take top-1 result → that chunk ID is the ground truth for this doc
 
    Returns a dict: { collection_name: [chunk_id_for_doc1, chunk_id_for_doc2, ...] }
    """
    result = {name: [] for name in COLLECTION_NAMES}
    embedding = embed_model.encode(answer).tolist()
 
    for doc_id in doc_ids:
        for col_name, col in collections.items():
            try:
                results = col.query(
                    query_embeddings=[embedding],
                    n_results=1,
                    where={"doc_id": doc_id},
                    include=["distances"]
                )
                ids = results.get("ids", [[]])[0]
 
                if ids:
                    result[col_name].append(ids[0])
                else:
                    print(f"  ⚠ [{col_name}] No chunks found for doc_id={doc_id}")
 
            except Exception as e:
                print(f"  ✗ [{col_name}] Query failed for doc_id={doc_id}: {e}")
 
    return result

In [ ]:
from tqdm import tqdm

print("\n── Loading data ──────────────────────────────────")
df = pd.read_parquet(PARQUET_PATH)
print(f"  Loaded {len(df)} questions from {PARQUET_PATH}")

print("\n── Connecting to ChromaDB ────────────────────────")
client      = chromadb.PersistentClient(path=CHROMA_PATH)
collections = load_collections(client)

if not collections:
    raise RuntimeError("No collections loaded. Check CHROMA_PATH and collection names.")

print("\n── Loading embedding model ───────────────────────")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
print(f"  ✓ Loaded: {EMBED_MODEL_NAME}")

print("\n── Resolving chunk IDs ───────────────────────────")
ground_truth_chunk_ids = []

for i, row in tqdm(df.iterrows(), total=df.shape[0], desc="Processing rows"):
    answer  = row.get("ground_truth_answer", "")
    doc_ids = row.get("source_document_ids", [])

    # Normalise: could be a JSON string in some parquet files
    if isinstance(doc_ids, str):
        doc_ids = json.loads(doc_ids)

    if not answer or not doc_ids:
        print(f"    ⚠ Skipping — missing answer or source_document_ids")
        ground_truth_chunk_ids.append({n: [] for n in COLLECTION_NAMES})
        continue

    ids_dict = resolve_chunk_ids_for_question(
        answer, doc_ids, collections, embed_model
    )
    ground_truth_chunk_ids.append(ids_dict)

print("\n── Saving output ─────────────────────────────────")
df["ground_truth_chunk_ids"] = ground_truth_chunk_ids

In [ ]:
# ── Quick sanity check ──
sample = df["ground_truth_chunk_ids"].iloc[0]
print("\n── Sample output (row 0) ─────────────────────────")
print(json.dumps(sample, indent=2))
print("\nDone ✓")

In [ ]:
print("\n── Saving output ─────────────────────────────────")
df["ground_truth_chunk_ids"] = ground_truth_chunk_ids
df.to_parquet(OUTPUT_PATH, index=False)
print(f"  ✓ Saved to {OUTPUT_PATH}")

In [1]:
import pandas as pd

questions_df = pd.read_parquet("../../data/processed/questions/questions_with_chunk_ids.parquet")

# Export to JSON Lines format
questions_df.to_json('../../data/processed/questions/questions.jsonl', orient='records', lines=True)